# 4 · Costs and the accuracy of the delivered question

Compare three actual implementations on the **same 600 rays and three targets**:

1. Pixels with the correct L2 mass matrix (the continuum-inconsistent baseline).
2. A fully finite, conforming H2 Galerkin problem in dense kernel translates.
3. Native H2 data-space Gram assembly.

The last two approximate the **same objective**. Pixel/L2 changes the comparison
geometry, so a fast pixel solve is not an accuracy-equivalent substitute. The
comparison does not benchmark every possible finite-element or iterative method.

We measure fresh assembly, metric elimination/Gram formation and factor/solve
costs. Imports and the common synthetic data generation are excluded for all.
Reference and accuracy diagnostics are outside the timed regions. No conclusion
about a winner is assumed in advance.

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib, json, os, socket, subprocess, sys, time
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import study as F
S=F.S
ex=F.Experiment(length_scale=.1)  # shared by both H2 routes; edit consistently in all notebooks
OUT=Path('output');OUT.mkdir(exist_ok=True)
COLORS=('#2563eb','#dc2626','#7c3aed')
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.2,'font.size':11})
print('host:',socket.gethostname(),'Python:',sys.executable)
print('source commit:',subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())
print('original problem SHA256:',hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
print('600 original rays; one fixed noise draw; sigma =',S.P.SIGMA)
for label,c,t in zip(F.LABELS,ex.centres,ex.truth):print(label,c,'radius',ex.radius,'true disk mean',f'{t:.9f}')
print('Fixed normalized trade-off lambda =',ex.lam,'; H2 length scale =',ex.length_scale)
import platform, scipy, pygeoinf0 as pg
from threadpoolctl import threadpool_info
runtime=dict(host=socket.gethostname(),python=sys.version,numpy=np.__version__,scipy=scipy.__version__,
             platform=platform.platform(),cpu_count=os.cpu_count(),package_source=pg.__file__,
             threadpools=threadpool_info(),targets=ex.centres,radius=ex.radius,lambda_value=ex.lam,length_scale=ex.length_scale,
             study_sha256=hashlib.sha256(Path(F.__file__).read_bytes()).hexdigest(),
             helper_sha256=hashlib.sha256(Path(S.__file__).read_bytes()).hexdigest(),
             problem_sha256=hashlib.sha256(Path(S.P.__file__).read_bytes()).hexdigest())
F.save(OUT/'runtime.json',runtime)
print('BLAS thread counts:',[v['num_threads'] for v in runtime['threadpools']])


## Method and timing controls

One process, one BLAS thread, fresh problem objects for each repetition. Native
Gram caching within a single solve is allowed; no previous method's assembled
Gram is supplied to another timed method. One common data-space factor handles
all three targets plus normalization. We report all raw repetitions and medians.

“Matrix payload” sums named major algebra arrays. It excludes workspaces,
allocator behavior, kernel objects and cached copies; **it is not peak RAM**.
The native formulation still has an N-by-N Gram here, while the dense Galerkin
realization also assembles a trial-by-trial Sobolev mass matrix.

In [ ]:
repeats=int(os.environ.get('SOLA_PART1_BENCH_REPEATS','3'))
ref,Xref,tau,reference=F.native_reference(ex,24,32)
records=[]
settings=[('pixels',n) for n in (80,320,1280)]
settings += [('H2 Galerkin',n) for n in (17,33,65) if n<=int(os.environ.get('SOLA_PART1_MAX_TRIAL','65'))]
settings += [('H2 native',n) for n in (10,16,24)]
# A small untimed BLAS warm-up is shared by every method.
la.cho_solve(la.cho_factor(np.eye(5)),np.ones(5))
for method,level in settings:
    for repeat in range(repeats):
        if method=='pixels':
            system=F.pixel_system(ex,level);penalty=ex.lam*ex.target_l2_norm2
        elif method=='H2 Galerkin':
            system=F.galerkin_system(ex,level,24,32);penalty=tau
        else:
            system=F.native_system(ex,level,32);penalty=tau
        X,row=F.solve_system(system,ex,penalty)
        row.update(method=method,level=level,repeat=repeat,
                   dimension=system['dimension'],**F.compare_to_reference(X,ref,Xref))
        records.append(row)
        print(method,level,'repeat',repeat,'seconds',round(row['total_seconds'],4),
              'relative distance to H2 reference',f"{row['weight_error']:.3e}",flush=True)
        del system
summary=[]
for method,level in settings:
    selected=[r for r in records if r['method']==method and r['level']==level]
    row=dict(method=method,level=level,dimension=selected[0]['dimension'],
             median_seconds=float(np.median([r['total_seconds'] for r in selected])),
             min_seconds=float(min(r['total_seconds'] for r in selected)),
             max_seconds=float(max(r['total_seconds'] for r in selected)),
             matrix_payload_MiB=selected[0]['matrix_bytes']/1024**2,
             weight_error=selected[0]['weight_error'],
             delivered_functional_error=selected[0]['delivered_functional_error'],
             timings={k:float(np.median([r['timings'][k] for r in selected])) for k in selected[0]['timings']})
    summary.append(row)
F.save(OUT/'benchmark.json',dict(records=records,summary=summary,repeats=repeats,
       host=socket.gethostname(),threads={k:os.environ.get(k) for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS')},
       reference=reference,accuracy_metric='relative data weights and H2 delivered-functional distance'))
import csv
with (OUT/'benchmark.csv').open('w',newline='') as f:
    names=[k for k in summary[0] if k!='timings'];w=csv.DictWriter(f,fieldnames=names);w.writeheader()
    w.writerows({k:r[k] for k in names} for r in summary)
print('\n method       level    seconds     matrix MiB    weight error')
for r in summary:print(f"{r['method']:12s} {r['level']:5d} {r['median_seconds']:11.4f} "
                      f"{r['matrix_payload_MiB']:12.2f} {r['weight_error']:14.4e}")


In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4),constrained_layout=True)
colours={'pixels':'#b45309','H2 Galerkin':'#2563eb','H2 native':'#059669'}
for method in colours:
    selected=[r for r in summary if r['method']==method]
    ax[0].loglog([r['level'] for r in selected],[r['median_seconds'] for r in selected],'o-',label=method,color=colours[method])
    ax[1].loglog([r['median_seconds'] for r in selected],np.maximum([r['delivered_functional_error'] for r in selected],1e-16),'o-',color=colours[method])
    ax[2].loglog([r['level'] for r in selected],[r['matrix_payload_MiB'] for r in selected],'o-',color=colours[method])
ax[0].legend(fontsize=8);ax[0].set_ylabel('median total seconds')
ax[1].set_xlabel('median total seconds');ax[1].set_ylabel('H2 delivered-question distance')
ax[1].set_title('Pixels: diagnostic distance, not same-objective error')
ax[2].set_ylabel('named matrix payload MiB (not peak RAM)')
for a in (ax[0],ax[2]):a.set_xlabel('resolution level (method-specific)')
plt.show()
fig,ax=plt.subplots(figsize=(12,4),constrained_layout=True)
labels=[f"{r['method']}\n{r['level']}" for r in summary];bottom=np.zeros(len(summary))
for key,label in [('data_assembly','data / trial assembly'),('target_assembly','target evaluations'),('metric_and_gram','metric elimination / Gram'),('solve','factor and solve')]:
    val=np.array([r['timings'][key] for r in summary]);ax.bar(labels,val,bottom=bottom,label=label);bottom+=val
ax.set_ylabel('median stage seconds');ax.legend(fontsize=8);plt.show()


## Compare costs at a declared accuracy

For the two H2 methods, use a relative weight-error threshold of $10^{-3}$ against
the fixed 24-node/32-chord reference. This threshold is a numerical study setting,
not an uncertainty interval. The separate target-quadrature check in notebook 3
bounds what can reasonably be inferred from that reference agreement.

For pixels, the reference distance describes how its selected question differs
from the chosen H2 question. It is **not** a pixel discretization error estimate.

In [ ]:
tolerance=1e-3
for method in ('H2 Galerkin','H2 native'):
    eligible=[r for r in summary if r['method']==method and r['weight_error']<=tolerance]
    if eligible:
        best=min(eligible,key=lambda r:r['median_seconds'])
        print(method,': cheapest measured setting meeting tolerance:',best['level'],
              'median seconds:',best['median_seconds'],'relative error:',best['weight_error'])
    else:print(method,': no measured setting met tolerance; do not invent an equal-accuracy speedup.')
if (OUT/'pixels.json').exists() and (OUT/'galerkin.json').exists():
    px=F.load(OUT/'pixels.json')['rows'][-1];ga=F.load(OUT/'galerkin.json')['rows'][-1]
    print('\nFinest pixel estimates:',px['estimate'])
    print('Finest Galerkin estimates:',ga['estimate'])
print('Native continuum estimates:',reference['estimate'])
print('Requested target truths:',ex.truth)


## Compare the actual questions with a common physical test

Use the same moving, unit-peak anomaly as in notebooks 1-3. Each map evaluates
the actual transformed ray data $XGq_z$; none is a pixel density or projected
Galerkin field. Physical rays and target disks remain exactly the same.

For $q_z=K_H(\cdot,z)/K_H(z,z)$ these are rescaled continuum Riesz functions,
but the interpretation is simply the predicted response to a known test model.
The display probe is shared across all methods, including pixel/L2; it does not
change their different matching objectives. This figure is outside all timings.
It uses the finest saved pixel and Galerkin weights from notebooks 1 and 2;
the plotted pixel level may differ from the benchmark's timed settings.

In [ ]:
# physical-probe: comparison
import probe_views as V
probes=V.response_grid(ex,OUT)
with np.load(OUT/'pixel_weights.npz') as saved:
    px_level=int(saved['grids'][-1]);px_weights=saved['weights'][-1]
with np.load(OUT/'galerkin_weights.npz') as saved:
    ga_level=int(saved['grids'][-1]);ga_weights=saved['weights'][-1]
V.plot_response_evolution(ex,probes,[px_weights[0],ga_weights[0],Xref[0]],
    [f'Pixels {px_level} x {px_level}',f'H2 Galerkin {ga_level} x {ga_level}','Native H2'],
    'One physical test family: the actual delivered questions versus the disk target');plt.show()
print('Responses to constant models:')
for label,W in [('pixels',px_weights),('H2 Galerkin',ga_weights),('native',Xref)]:
    print(label,W@ex.v)


## End of part 1

- The pixel example uses its mass matrix correctly. Its ray/L2 continuum
  interpretation still fails under refinement.
- A declared Sobolev space makes the observations and the matching criterion
  well-defined. Conforming finite discretization and native continuum assembly
  approach the same question.
- Actual costs depend on representation, accuracy, data count, number of targets
  and implementation. This benchmark is evidence for these settings and this dense
  trial basis, not a universal performance ordering.
- Numerical convergence does not remove target mismatch or assert prior bounds.

This notebook sequence is the reviewable first part. Later presentation sections
and the Slidev deck are deliberately left for the author's next instructions.